# ML in Rust: benchmark notebooks (start here)

This project races ML algorithms built several ways against scikit-learn, to answer one question: **where does compiled code actually beat scikit-learn, and where is scikit-learn already as fast as the machine allows?**

It began as the first version of [autoalgo](https://github.com/herodthegreater/autoalgo) (tag `speed-study-v0.2`) and was split out once the results changed autoalgo's direction.

| Notebook | Algorithm | Bottleneck it tests | Engines raced |
|---|---|---|---|
| `01_knn` | k-nearest neighbors | matrix math, then picking the k smallest (memory) | numpy, rust, rust-blas, mlx |
| `02_kmeans` | k-means | matrix math repeated every iteration, plus summing rows per cluster | numpy, rust, mlx |
| `03_ridge` | ridge regression | one big matrix multiply, then a tiny solve | numpy, mlx |
| `04_pca` | PCA | one big matrix multiply, then a tiny eigen-solve, *or* a full SVD depending on the solver | numpy, mlx |
| `05_decision_tree` | decision tree | branchy, integer counting: no matrix math at all | numpy, rust |
| `06_summary` | all of the above | the scoreboard, across every machine that has results | |

**What the engine names mean**

- `sklearn`: scikit-learn's own estimator, default settings, all cores. The baseline.
- `numpy`: built *on top of* existing libraries. Python plus NumPy, which hands matrix math to its BLAS (Apple Accelerate on macOS, OpenBLAS on Linux).
- `rust`: compiled Rust (`src/`), every core, GIL released, built for this machine's CPU. k-means also uses a tuned matrix multiply.
- `rust-blas` (KNN only): Rust that hands the distance math to a tuned multiply: Apple Accelerate on macOS (which can use Apple's hidden matrix units), the pure-Rust `matrixmultiply` crate elsewhere.
- `mlx`: pure Python on the Apple GPU through MLX. Skipped on machines without MLX.

Some algorithms have no Rust or MLX engine on purpose. Each notebook says why.

## How to run these yourself

Easiest: from the project folder, run `./run_all.sh`. It builds everything, runs the tests, executes every notebook, and saves this machine's numbers to `notebooks/results/<machine>/`.

Or by hand:
1. Open the `ML-in-rust-experiment` folder in your IDE and pick `.venv/bin/python` as the kernel.
2. If you change anything in `src/` (the Rust code), rebuild first: `.venv/bin/maturin develop --release`
3. Run the notebooks top to bottom. `06_summary` reads every machine's results.

For a fast smoke run on small data, use `QUICK=1 ./run_all.sh` (or set `QUICK = True` near the top of a notebook).

In [ ]:
import sys
sys.path.insert(0, ".")  # so bench_utils (next to this notebook) imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_in_rust as mr
from bench_utils import (machine_info, timeit, load_covtype, add_speedup,
                         save_results, line_chart, fmt_ms, only_available)

pd.set_option("display.float_format", "{:,.4f}".format)
%matplotlib inline
pd.Series(machine_info())

## Sanity check: every engine imports and agrees on a tiny problem

In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier as SkKNN

X, y = load_iris(return_X_y=True)
ref = SkKNN(5).fit(X, y).predict(X)
pd.DataFrame({
    backend: {"matches scikit-learn": bool((mr.KNeighborsClassifier(5, backend=backend).fit(X, y).predict(X) == ref).all())}
    for backend in mr.available_backends()
}).T